# Inspect products with RFI

Plot the sepctrum and spectrogram of products affected by RFI.

In [ ]:
%pip install aiohttp requests
%load_ext autoreload
%autoreload 2
    
from shapely.geometry import Polygon
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import os
import json
import yaml
from pathlib import Path

from processing import pol_mean, read_item
from plotting import fetch_quicklook, plot_product, pauli_rgb

In [ ]:
# Load the config.yaml to access the paths
cfg_file = Path("../config.yaml").resolve()
cfg = yaml.safe_load(cfg_file.read_text())

# Set the name of the path in the config.yaml
token_path = (cfg_file.parent / cfg["paths"]["token_file"]).resolve()
out_path = (cfg_file.parent / cfg["paths"]["output"]).resolve()
data_path = (cfg_file.parent / cfg["paths"]["data"]).resolve()

rfi_prod_path = out_path / cfg["files"]["rfi_products"]
gdf_path = out_path / cfg["files"]["gdf"]

print(f"Token Path: {token_path}")
print(f"Output Path: {out_path}")
print(f"List of RFI products Path: {rfi_prod_path}")
print(f"GeoDataframe of RFI products Path: {gdf_path}")

In [ ]:
# Load the list of products affected by RFI
with open(rfi_prod_path) as f:
    products_rfi = json.load(f)
    
# Load the GeoDataframe
gdf = gpd.read_parquet(gdf_path)

gdf.head()
print(len(gdf))
print(len(products_rfi))

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

if pathlib.Path(token_path).exists():
  with open(token_path, "rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Select which products to explore:
- From the products_rfi.json which are converted to a dataframe. As an example we select the unmitigated proucts with an iso_impact larger than 10%
- Select any of the hard-coded list
- Conventional filtering by bbox, dates etc.

In [ ]:
print(len(gdf))
print(len(products_rfi))
rows = [{"product_id": p["id"], "rfiCorrectionFlag": p["rfiCorrectionFlag"], **p["properties"], **p["fm"]}
        for p in products_rfi]
print(len(rows))
df_rfi = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["product_id"])
print(len(df_rfi))
for kind in ("pers", "iso"):
    df_rfi[f"{kind}_avgBW"] = pol_mean(df_rfi, kind, "avgBW")
    df_rfi[f"{kind}_linesPct"] = pol_mean(df_rfi, kind, "linesPct")
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100
    
df_selected = df_rfi[df_rfi["rfiCorrectionFlag"] == "false"]

PRCT_IMPACT = 10.0
n_before = len(df_selected)
df_selected = df_selected[df_selected["iso_impact"] >= PRCT_IMPACT]
print(f"discarded {n_before - len(df_selected)} of {n_before} products "
      f"(iso_impact < {PRCT_IMPACT} or missing), {len(df_selected)} kept")

In [ ]:
# Middle east
middle_east = ['BIO_S1_SCS__1S_20260513T150807_20260513T150827_T_G01_M03_C07_T039_F134_02_DRE0O0', 
               'BIO_S1_SCS__1S_20260513T150904_20260513T150924_T_G01_M03_C07_T039_F137_02_DRDZV9', 
               'BIO_S1_SCS__1S_20260513T150826_20260513T150846_T_G01_M03_C07_T039_F135_02_DRE0HI', 
               'BIO_S3_SCS__1S_20260829T150437_20260829T150457_T_G01_M04_C05_T039_F107_02_DWY0RR',
               'BIO_S3_SCS__1S_20260829T150359_20260829T150419_T_G01_M04_C05_T039_F105_02_DWY14T']
# Ecuador
ecuador = [
    "BIO_S3_SCS__1S_20260829T233255_20260829T233313_T_G01_M04_C05_T044_F162_02_DWYO99",
    "BIO_S3_SCS__1S_20260829T233236_20260829T233257_T_G01_M04_C05_T044_F161_02_DWYO8H",
    "BIO_S3_SCS__1S_20260829T233217_20260829T233238_T_G01_M04_C05_T044_F160_02_DWYO38",
    "BIO_S3_SCS__1S_20260829T233158_20260829T233219_T_G01_M04_C05_T044_F159_02_DWYO4X",
    "BIO_S3_SCS__1S_20260829T233139_20260829T233200_T_G01_M04_C05_T044_F158_02_DWYO5W",
    "BIO_S3_SCS__1S_20260829T233120_20260829T233141_T_G01_M04_C05_T044_F157_02_DWYO5M",
    "BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C",
    "BIO_S3_SCS__1S_20260829T233042_20260829T233103_T_G01_M04_C05_T044_F155_02_DWYOAL",
    "BIO_S3_SCS__1S_20260829T233023_20260829T233044_T_G01_M04_C05_T044_F154_02_DWYO9B",
    "BIO_S3_SCS__1S_20260829T233004_20260829T233025_T_G01_M04_C05_T044_F153_02_DWYOAY",
]

selected = [
    # Set 1, Region 1
    "BIO_S1_SCS__1S_20260915T104023_20260915T104044_T_G01_M05_C01_T021_F293_02_DXTLHJ",
    "BIO_S1_SCS__1S_20260918T104025_20260918T104046_T_G01_M05_C02_T021_F293_02_DXZAG1",
    "BIO_S1_SCS__1S_20260921T104028_20260921T104048_T_G01_M05_C03_T21_F293_02_DY4MWQ",
    "BIO_S1_SCS__1S_20260924T104030_20260924T104050_T_G01_M05_C04_T021_F293_02_DYA6ZC",
    "BIO_S1_SCS__1S_20260927T104032_20260927T104052_T_G01_M05_C05_T021_F293_02_DYFQMZ",
    "BIO_S1_SCS__1S_20260930T104034_20260930T104055_T_G01_M05_C06_T021_F293_02_DYLATM",
    "BIO_S1_SCS__1S_20260915T104101_20260915T104122_T_G01_M05_C01_T021_F295_02_DXTLHG",
    "BIO_S1_SCS__1S_20260918T104103_20260918T104124_T_G01_M05_C02_T021_F295_02_DXZAHN",
    "BIO_S1_SCS__1S_20260921T104106_20260921T104126_T_G01_M05_C03_T021_F295_02_DY4MZ4",
    "BIO_S1_SCS__1S_20260924T104108_20260924T104128_T_G01_M05_C04_T021_F295_02_DYA6ZD",
    "BIO_S1_SCS__1S_20260927T104110_20260927T104130_T_G01_M05_C05_T021_F295_02_DYFQOH",
    "BIO_S1_SCS__1S_20260930T104112_20260930T104133_T_G01_M05_C06_T021_F295_02_DYLAZI",
    # Set 1, Region 2
    "BIO_S1_SCS__1S_20260916T220549_20260916T220609_T_G01_M05_C01_T043_F184_02_DXW72Z",
    "BIO_S1_SCS__1S_20260919T220551_20260919T220611_T_G01_M05_C02_T043_F184_02_DY1SIO",
    "BIO_S1_SCS__1S_20260922T220553_20260922T220614_T_G01_M05_C03_T043_F184_02_DY79AQ",
    "BIO_S1_SCS__1S_20260925T220555_20260925T220616_T_G01_M05_C04_T043_F184_02_DYCSA3",
    "BIO_S1_SCS__1S_20260928T220558_20260928T220618_T_G01_M05_C05_T043_F184_02_DYICLW",
    # Set 2, Region 1
    "BIO_S1_SCS__1S_20260915T212528_20260915T212549_T_G01_M05_C01_T028_F160_02_DXUDDE",
    "BIO_S1_SCS__1S_20260918T212530_20260918T212551_T_G01_M05_C02_T028_F160_02_DY00H6",
    "BIO_S1_SCS__1S_20260921T212532_20260921T212553_T_G01_M05_C03_T028_F160_02_DY5EBB",
    "BIO_S1_SCS__1S_20260924T212535_20260924T212555_T_G01_M05_C04_T028_F160_02_DYAYBQ",
    "BIO_S1_SCS__1S_20260927T212537_20260927T212557_T_G01_M05_C05_T028_F160_02_DYGHV7",
    # Set 2, Region 2
    "BIO_S1_SCS__1S_20260915T090619_20260915T090640_T_G01_M05_C01_T020_F306_02_DXTDVD",
    "BIO_S1_SCS__1S_20260918T090622_20260918T090642_T_G01_M05_C02_T020_F306_02_DXZ169",
    "BIO_S1_SCS__1S_20260921T090624_20260921T090644_T_G01_M05_C03_T020_F306_02_DY4IY2",
    "BIO_S1_SCS__1S_20260924T090626_20260924T090646_T_G01_M05_C04_T020_F306_02_DYA1LO",
    "BIO_S1_SCS__1S_20260927T090628_20260927T090648_T_G01_M05_C05_T020_F306_02_DYFMCQ",
    "BIO_S1_SCS__1S_20260930T090630_20260930T090651_T_G01_M05_C06_T020_F306_02_DYL677",
]

bbox_south_america = [-81.5, -56.0, -34.0, 13.0]       # lists of floats, not strings
bbox_antarctica    = [-180, -90, 180, -60]

bbox = None # Set to valid bbox if filter with it
# Which products to load: "ids" (hard-coded list `selected`), "top_impact" (N largest iso_impact in products_rfi.json)
# or "threshold" (unmitigated products with iso_impact >= PRCT_IMPACT, from df_selected)
SELECT = "threshold"
N = 20
if SELECT == "top_impact":
    ids = df_rfi.nlargest(N, "iso_impact")["product_id"].to_list()
elif SELECT == "random":
    ids = df_rfi.sample(N)["product_id"].to_list()
elif SELECT == "threshold":
    ids = df_selected["product_id"].to_list()
else:
    ids = ecuador
print(len(ids))
if bbox is not None:
    search = catalog.search(
        method="GET",
        collections=["BiomassLevel1a"],
        datetime=["2026-01-01T00:00:00Z", "2026-07-29T23:33:00Z"],
        bbox=bbox,
        max_items=len(ids),
    )
    items = list(search.items())
    print(f"{len(items)} items accessed (max_items=10), {search.matched()} matched the query.")
    data = search.item_collection_as_dict()
else:
    B = 100
    searches = [catalog.search(method="POST", ids=ids[i:i + B], limit=B) for i in range(0, len(ids), B)]
    items = [it for s in searches for it in s.items()]
    missing = set(ids) - {it.id for it in items}
    print(f"{len(items)} of {len(ids)} ids found, {len(missing)} missing.")
    # convert to df for easy visualisation
    data = {"type": "FeatureCollection",
        "features": [f for s in searches for f in s.items_as_dicts()]}

In [ ]:
df = pd.json_normalize(data, record_path=["features"])[
    [
        "id",  # unique id
        "properties.eopf:datatake_id",  # not unique id
        "properties.product:type",
        "properties.updated",
        "properties.sat:absolute_orbit",
        "properties.sat:orbit_state",
        # "assets.thumbnail.href",
        "assets.enclosure_kml.href",
        "assets.quicklook_jpg.href",
        "assets.enclosure_i_abs_tiff.href",
        "assets.enclosure_i_phase_tiff.href",
        "assets.product.href",
    ]
]

# Renaming the assets for
df.rename(
    columns={
        "properties.eopf:datatake_id": "dt_id",
        "properties.product:type": "product_type",
        "properties.updated": "last_modified",
        "properties.sat:absolute_orbit": "abs_orbit",
        "properties.sat:orbit_state": "orbit_state",
        # "assets.thumbnail.href": "thumbnail",
        "assets.enclosure_kml.href": "quicklook_kml",
        "assets.quicklook_jpg.href": "quicklook_jpg",
        "assets.enclosure_i_abs_tiff.href": "abs_product",
        "assets.enclosure_i_phase_tiff.href": "phase_product",
        "assets.product.href": "zipped_product",
    },
    inplace=True,
)

df.sort_values(by="id", ascending=True, ignore_index=True, inplace=True)
df

In [ ]:
def read_gpd_streaming(url, token):
    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
    with fs.open(url, "rb") as f:
        kml_data = f.read()  # get file from HTTPS using token
        root = ET.fromstring(kml_data)  # parse kml
        ns = {
            "kml": "http://www.opengis.net/kml/2.2",
            "gx": "http://www.google.com/kml/ext/2.2",
        }  # define namespaces
        latlonquads = root.findall(".//gx:LatLonQuad", ns)  # get gx:LatLonQuad elements

        polygons = []
        for quad in latlonquads:
            coords_elem = quad.find("coordinates")  # find coords
            if coords_elem is None:
                continue
            coords_text = coords_elem.text.strip()
            coord_pairs = coords_text.split()
            coords = [tuple(map(float, pair.split(","))) for pair in coord_pairs]
            if len(coords) >= 4:  # should be a closed polygon
                polygons.append(Polygon(coords))

        gdf = gpd.GeoDataFrame(
            {"name": "unnamed", "geometry": polygons}, crs="EPSG:4326"
        )

        return gdf
        
def read_rio_streaming(url, token, subset=False):
    """
    Stream a remote raster file over HTTPS using a bearer token and read
    its band data with rasterio.

    The function:
    - Authenticates using an HTTP Authorization header
    - Opens the raster file directly from a remote stream
    - Reads all raster bands into memory
    - Optionally reads only a specified pixel window (subset)
    - Accesses Ground Control Points (GCP-based georeferencing)

    Parameters
    ----------
    url : str
        HTTPS URL pointing to a raster file (e.g., GeoTIFF).
    token : str
        Bearer token used for HTTP authentication.
    subset : tuple or False, optional
        Optional pixel window defined as
        (col_offset, row_offset, width, height).
        If False, the full raster is read.

    Returns
    -------
    src : rasterio.io.DatasetReader
        Rasterio dataset object (closed after reading).
    bands : list of numpy.ndarray
        List of arrays containing raster band data.
    """

    # Create an authenticated HTTPS filesystem for streaming access
    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})

    # Open the remote raster file as a binary stream
    print("Fetching file")
    with fs.open(url, "rb") as f:
        with rio.open(f) as src:
            print("File Openned")
            if subset:
                # Define a pixel window to read only part of the raster
                window = Window(
                    col_off=subset[0],
                    row_off=subset[1],
                    width=subset[2],
                    height=subset[3],
                )

                # Read each band within the specified window
                bands = [src.read(i + 1, window=window) for i in range(src.count)]

                # Retrieve ground control points (GCP-based georeferencing)
                gcps = src.get_gcps()
            else:
                # Read full raster bands into memory
                bands = [src.read(i + 1) for i in range(src.count)]

                # Retrieve ground control points (GCP-based georeferencing)
                gcps = src.get_gcps()

        return src, bands

In [ ]:
gdfs = []
for i, url in df["quicklook_kml"].items():
    g = read_gpd_streaming(url, access_token)
    g["item"] = i
    g["name"] = items[i].id
    gdfs.append(g)
allg = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

c = allg.geometry.union_all().centroid
m = folium.Map(location=[c.y, c.x], zoom_start=4, tiles="OpenStreetMap")
folium.GeoJson(allg, tooltip=folium.GeoJsonTooltip(fields=["name"])).add_to(m)
m.fit_bounds([[allg.total_bounds[1], allg.total_bounds[0]], [allg.total_bounds[3], allg.total_bounds[2]]])
m

In [ ]:
by_id = {it.id: it for it in items}
items_selected = [by_id[i] for i in ids if i in by_id]
print(f"{len(items_selected)} products to plot")

### Plot with streaming

In [ ]:
for item in items_selected:
    _, bands = read_rio_streaming(item.assets["enclosure_i_abs_tiff"].href, access_token, subset=False)
    _, phase = read_rio_streaming(item.assets["enclosure_i_phase_tiff"].href, access_token, subset=False)
    slc = np.asarray(bands) * np.exp(1j * np.asarray(phase))      # (pol, azimuth, range)

    plot_product(slc, item.id, fetch_quicklook(item, token=access_token))
    del slc, bands, phase                                          # a full SCS is large

## Download and plot

In [ ]:
import requests, numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

KEYS = ["enclosure_annot_xml", "enclosure_i_abs_tiff", "enclosure_i_phase_tiff", "quicklook_jpg"]

def download_file(href, dest, token):
    if dest.exists():                                   # skip already downloaded files
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")       # avoid half-written files looking complete
    with requests.get(href, headers={"Authorization": f"Bearer {token}"}, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
    tmp.rename(dest)
    return dest

def download_item(item, token=None):
    token = token or access_token
    out = data_path / item.id; out.mkdir(parents=True, exist_ok=True)
    return {k: download_file(item.assets[k].href, out / Path(item.assets[k].href).name, token) for k in KEYS}

# 1) download all, in parallel
def safe(item):
    try:
        return item.id, download_item(item), None
    except Exception as e:
        return item.id, None, repr(e)

with ThreadPoolExecutor(max_workers=12) as ex:
    res = list(tqdm(ex.map(safe, items_selected), total=len(items_selected)))
paths_by_id = {i: p for i, p, e in res if p}
failed = {i: e for i, p, e in res if e}
print(len(paths_by_id), "ok,", len(failed), "failed")

In [ ]:
# 2) read one product on demand (not all at once, a full SCS is large)
POLS = ["HH", "HV", "VH", "VV"]   # assumed band order, check src.descriptions or the annotation XML

for item in items_selected:
    if item.id not in paths_by_id:
        continue                                                   # failed download
    slc = read_item(item.id, paths_by_id)
    plot_product(slc, item.id, fetch_quicklook(item, paths_by_id[item.id]["quicklook_jpg"]), out_dir = f"{out_path}/{item.id}")
    del slc

In [ ]:
ids_pauli = ["BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C"]
MAX_PX = 4000  # decimation as in plot_result: box-average by n // MAX_PX per axis (None = full resolution)

for item in items_selected:
    if item.id not in ids_pauli or item.id not in paths_by_id:  # only the listed products that are downloaded
        continue
    slc = read_item(item.id, paths_by_id)  # (4, R, C), POLS order
    if slc is None:
        continue
    rgb = pauli_rgb(slc, max_px=MAX_PX)  # same Pauli image as the panels of plot_result
    del slc
    plt.imsave(f"{out_path}/pauli_{item.id}.png", rgb)  # one image pixel per SLC pixel: no axes, no margins
    plt.figure(figsize=(4, 14))
    plt.imshow(rgb, aspect="auto")
    plt.axis("off")
    plt.show()
    del rgb
